<h2 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
Purpose of the Model 
</h2>

This model recognizes hand signs and gestures from right-hand landmark data to identify letters, numbers, and custom words in sign language.  

<p><img style="float:right; margin:50px; padding:20px; max-height:240px; margin-top:-30px" src="https://www.lighthouseonline.com/wp-content/uploads/2022/12/sign-language.jpg.webp"></p>

It has **two main parts**:  
1. **Building the Sign Recognition Model** : classifying hand gestures into letters, numbers, or custom words using normalized landmark features and K-Nearest Neighbors (KNN).  
2. **Data Augmentation & Normalization** : improving recognition accuracy by normalizing hand landmarks and generating additional samples for rare classes.

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
 Main Prediction
</h3>
<strong>Sign Class</strong> – the predicted letter, number, or custom word corresponding to the hand gesture.

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
How It Works
</h3>
The model analyzes normalized 3D hand landmarks, including:

- **Wrist and finger joint positions**
- **Relative distances and angles between landmarks**
- **Patterns in hand posture for specific letters, numbers, or words**

Using these features, the model predicts the correct sign class. Rare or underrepresented signs are augmented with slight variations to improve robustness.

<h2 style="
    /* background-color: transparent;  */
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
 Main Prediction
</h2>
<strong>Sign Class<strong> the predicted letter, number, or word corresponding to the hand gesture

<h3 style="
    /* background-color: transparent;  */
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
 Importing Libraries
</h3>

In [ ]:
# ·············································
# :           Import necessary libraries      :
# ·············································
import os
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
 Paths and File Locations
</h3>


In [ ]:
# ·············································
# :               paths                       :
# ·············································
EXISTING_DATA = "../data/landmark_data_right_hand.npz"
CUSTOM_WORDS_DIR = "../data/custom_words/"
COMBINED_OUTPUT = "../data/combined_landmark_data.npz"
MODEL_OUTPUT = "../models/knn_combined_model2.pkl"

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 450px;
">
 Data Augmentation for Rare Classes
</h3>


In [ ]:
# ·············································
# :         Data Augmentation                 :
# ·············································
def augment_landmarks(X, y, min_samples=50):
    """
    Oversample rare classes using small random perturbations.
    """
    unique_classes, counts = np.unique(y, return_counts=True)
    X_aug = [x for x in X]  # each row individually
    y_aug = [label for label in y]  # each label individually

    for cls, count in zip(unique_classes, counts):
        if count < min_samples:
            idxs = np.where(y == cls)[0]
            needed = min_samples - count
            for _ in range(needed):
                sample = X[np.random.choice(idxs)]
                # small random noise
                noise = np.random.normal(0, 0.02, size=sample.shape)
                X_aug.append(sample + noise)
                y_aug.append(cls)

    return np.array(X_aug), np.array(y_aug)

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 500px;
">
 Wrist-Centered Landmark Normalization
</h3>

In [ ]:
# ·············································
# :        Normalization of Landmarks         :
# ·············································
def normalize_landmarks(X):
    X_norm = np.copy(X)
    for i in range(X.shape[0]):
        landmarks = X[i].reshape(-1, 3)
        wrist = landmarks[0]  # assuming wrist is first landmark
        landmarks -= wrist
        max_dist = np.max(np.linalg.norm(landmarks, axis=1))
        landmarks /= max_dist
        X_norm[i] = landmarks.flatten()
    return X_norm

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 500px;
">
 Loading and Combining Datasets
</h3>

In [ ]:
# ·············································
# :        Load & combine datasets            :
# ·············································
def combine_datasets():
    existing_data = np.load(EXISTING_DATA)
    X_existing = existing_data["X"]
    y_existing = existing_data["y"]
    print(f"Existing data: {X_existing.shape[0]} samples, {len(np.unique(y_existing))} classes" ,flush=True)

    # Load custom words
    X_custom, y_custom = [], []
    print("\nLoading custom words data..." ,flush=True)
    for word_dir in os.listdir(CUSTOM_WORDS_DIR):
        word_path = os.path.join(CUSTOM_WORDS_DIR, word_dir)
        if os.path.isdir(word_path):
            for npz_file in os.listdir(word_path):
                if npz_file.endswith("_landmarks.npz"):
                    data = np.load(os.path.join(word_path, npz_file))
                    X_custom.append(data["X"])
                    y_custom.append(data["y"])

    if X_custom:
        X_custom = np.vstack(X_custom)
        y_custom = np.concatenate(y_custom)
        print(f"Custom words data: {X_custom.shape[0]} samples, {len(np.unique(y_custom))} classes" ,flush=True)
        X_combined = np.vstack([X_existing, X_custom])
        y_combined = np.concatenate([y_existing, y_custom])
    else:
        X_combined, y_combined = X_existing, y_existing
        print("No custom words data found.")

    # Normalize landmarks
    X_combined = normalize_landmarks(X_combined)
    # Augment rare classes
    X_combined, y_combined = augment_landmarks(X_combined, y_combined)

    # Save combined
    np.savez_compressed(COMBINED_OUTPUT, X=X_combined, y=y_combined)
    print(f"\nCombined dataset saved: {COMBINED_OUTPUT}" ,flush=True)
    print(f"Total samples after augmentation: {X_combined.shape[0]}" ,flush=True)
    print(f"Total classes: {len(np.unique(y_combined))}" ,flush=True)
    return X_combined, y_combined

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
 Train and Evaluate Model
</h3>

In [ ]:
# ·············································
# :               train model                 :
# ·············································
def train_combined_model():
    X, y = combine_datasets()

    print("\nTraining new model...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Compute sample weights to handle imbalance
    sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
    
    model = KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        metric="euclidean"
        )
    model.fit(X_train, y_train)  # remove sample_weight!


    y_pred = model.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Save model
    with open(MODEL_OUTPUT, "wb") as f:
        pickle.dump({
            "model": model,
            "X_train": X_train,
            "y_train": y_train,
            "all_classes": list(np.unique(y))
        }, f)
    print(f"\nModel saved to: {MODEL_OUTPUT}")
    return model

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
 Generate UI Configuration File
</h3>

In [ ]:
# ·············································
# :            update UI & config             :
# ·············································
def update_ui_config():
    data = np.load(COMBINED_OUTPUT)
    all_classes = sorted(np.unique(data["y"]).tolist())
    config = {
        "all_classes": all_classes,
        "letters_numbers": [c for c in all_classes if len(c) == 1],
        "words": [c for c in all_classes if len(c) > 1]
    }
    import json
    os.makedirs("config", exist_ok=True)
    with open("config/signs_config.json", "w") as f:
        json.dump(config, f, indent=2)
    print(f"\nUI configuration updated: config/signs_config.json")

<h3 style="
    background-color: #9d4edd;
    color: #ffffff; 
    font-weight: 500;
    border: 2px solid #9d4edd;
    border-radius: 8px;
    padding: 10px 15px; ;
    box-shadow: #9d4edd 0px 0px 8px;
    width: 400px;
">
 Main Pipeline Execution
</h3>

In [ ]:
# ·············································
# :                 main                      :
# ·············································
if __name__ == "__main__":
    train_combined_model()
    update_ui_config()

Existing data: 84501 samples, 36 classes

Loading custom words data...
Custom words data: 250 samples, 5 classes

Combined dataset saved: ../data/combined_landmark_data.npz
Total samples after augmentation: 84782
Total classes: 41

Training new model...

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        14
           2       1.00      0.86      0.92        14
           3       1.00      1.00      1.00        17
           4       0.99      1.00      0.99       261
           5       0.98      1.00      0.99        62
           6       1.00      0.67      0.80        15
           7       1.00      0.90      0.95        20
           8       0.96      1.00      0.98        45
           9       1.00      0.88      0.94        17
   Calm Down       1.00      1.00      1.00        10
       Hello       1.00      0.90      0.95        10
      Mother       